# Atoms_Engine.ipynb — Master Controller
### NeuralAtoms Physical Intelligence Factory

**Hardware**: Google Colab T4 GPU  
**Source of Truth**: Local NVMe SSD (`~/Desktop/Neuralatoms` via Local Runtime)

---

In [ ]:
# Task 3.1: Mount Local Workspace & Configure Environment
import os
import sys
from pathlib import Path

# Set local project root as the primary source of truth
PROJECT_ROOT = Path("/content/Neuralatoms") if os.path.exists("/content/Neuralatoms") else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Industrial Workspace Active: {PROJECT_ROOT}")
print(f"Python Path: {sys.path[:1]}")

In [ ]:
# Task 3.2: GPU Warmup & Kinematics Diagnostic
import torch

def gpu_warmup():
    if not torch.cuda.is_available():
        raise RuntimeError("CRITICAL: T4 GPU NOT DETECTED. Execution Halted.")
    
    device = torch.device("cuda")
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    
    # Warm up: Matrix multiplication to prime CUDA cores
    a = torch.randn(2048, 2048).to(device)
    b = torch.randn(2048, 2048).to(device)
    for _ in range(10): # Iterative warmup
        c = torch.matmul(a, b)
    
    torch.cuda.synchronize()
    print("GPU Warmup Complete — CUDA Cores Optimized.")

gpu_warmup()

In [ ]:
# Task 3.3: The "Shadow" Robot Test
import mujoco
import numpy as np

def shadow_robot_test():
    print("Initializing Shadow Robot Physical Validation...")
    
    # 1. Load Humanoid Template from Compartment
    xml_path = PROJECT_ROOT / "Compartments" / "Dynamics" / "humanoid_template.xml"
    if not xml_path.exists():
        # Fallback to internal MuJoCo humanoid if template missing during first setup
        model = mujoco.MjModel.from_xml_string("<mujoco><worldbody><body name='pelvis'><joint type='free'/><geom size='0.1' type='sphere'/><body name='thigh'><joint name='hip' type='ball'/><geom size='0.1' type='capsule'/></body></body></worldbody></mujoco>")
    else:
        with open(xml_path, "r") as f:
            model = mujoco.MjModel.from_xml_string(f.read())

    data = mujoco.MjData(model)
    
    # 2. Apply 50Nm test torque to hip joint
    # Assuming hip is joint 1 (after freejoint)
    test_torque = 50.0 # Nm
    data.qfrc_applied[6] = test_torque # Ball joint address
    
    # 3. Step Physics
    mujoco.mj_step(model, data)
    
    # 4. Record acceleration (rad/s^2)
    accel = data.qacc[6]
    print(f"Applied Torque: {test_torque}Nm")
    print(f"Measured Acceleration: {accel:.4f} rad/s^2")
    
    # 5. Persist Result to Ledger
    ledger_path = PROJECT_ROOT / "Compartments" / "Ledger" / "shadow_test.atoms.npz"
    np.savez_compressed(ledger_path, torque=test_torque, accel=accel)
    print(f"Result Recorded to Local Ledger: {ledger_path.name} ✓")

shadow_robot_test()